In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import friedmanchisquare
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()
check_gpu()

print("✓ All libraries imported successfully")
print(f"✓ Scipy version: {stats.__version__ if hasattr(stats, '__version__') else 'not available'}")

In [ ]:
# Reload the module to get the latest changes (including calculate_mape)
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully with calculate_mape function")

# Friedman Test for Model Comparison
## Statistical Analysis of MAPE Results from Multiple Training Iterations

**Purpose:** Compare BiLSTM, BiGRU, LSTM, and GRU models using the Friedman non-parametric test across multiple training iterations (5 and 10 runs).

**Experiments Tested:**
- Experiment 1 (Exp1_80_20): Same-stock prediction
- Experiment 2 (Exp2_80_20): Cross-stock prediction

**Metrics:** MAPE (Mean Absolute Percentage Error) - lower is better


## Setup Configuration

In [ ]:
# Configuration
DATA_DIR = 'dataset'
TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'

# Create output directories
os.makedirs('results', exist_ok=True)
os.makedirs('figures/friedman_test', exist_ok=True)

# Iterations to run
ITERATIONS_5 = 5
ITERATIONS_10 = 10

print(f"Configuration:")
print(f"  Train Ratio: {TRAIN_RATIO} ({RATIO_LABEL})")
print(f"  Iterations: {ITERATIONS_5} and {ITERATIONS_10}")
print(f"  Models: {MODEL_TYPES}")
print(f"  Stocks: {STOCKS}")

## Load Data

In [ ]:
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\n✓ All daily data loaded!")

## Helper Functions for Training and Metric Collection

In [ ]:
def train_single_model(X_train, y_train, X_test, y_test, model_type, iteration):
    """Train a single model and return MAPE result"""
    try:
        set_seed(RANDOM_SEED + iteration)  # Different seed for each iteration
        
        # Build model using standard architecture
        model = build_model(model_type, lookback=LOOKBACK, units=UNITS, dropout=DROPOUT_RATE)
        
        # Train with NO early stopping - just the checkpoint for best model
        model.fit(
            X_train, y_train,
            validation_split=0.1,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        
        # Predict
        y_pred_scaled = model.predict(X_test, verbose=0).flatten()
        y_pred = proportion_inverse_scale(y_pred_scaled)
        y_true = proportion_inverse_scale(y_test)
        
        # Calculate MAPE
        mape = calculate_mape(y_true, y_pred)
        
        tf.keras.backend.clear_session()
        
        return mape
    except Exception as e:
        print(f"    Error training {model_type}: {str(e)}")
        return np.nan

def run_multiple_iterations(exp_label, n_iterations, is_same_stock=True):
    """
    Run multiple training iterations and collect MAPE results.
    
    Returns:
        DataFrame with columns: Iteration, Stock, Model, MAPE
        For Exp2, also includes Train_Stock and Test_Stock columns
    """
    results = []
    
    print(f"\n{'='*70}")
    print(f"Running {n_iterations} iterations for {exp_label}")
    print(f"{'='*70}\n")
    
    if is_same_stock:
        # Experiment 1: Same-stock prediction
        for iteration in range(n_iterations):
            print(f"Iteration {iteration + 1}/{n_iterations}")
            for stock in STOCKS:
                X_train, y_train, X_test, y_test, _ = prepare_same_stock_data(
                    daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK
                )
                
                for model_type in MODEL_TYPES:
                    mape = train_single_model(X_train, y_train, X_test, y_test, model_type, iteration)
                    results.append({
                        'Iteration': iteration + 1,
                        'Stock': stock,
                        'Model': model_type,
                        'MAPE': mape
                    })
                    print(f"  {stock} - {model_type}: MAPE = {mape:.4f}%" if not np.isnan(mape) else f"  {stock} - {model_type}: FAILED")
            print()
    else:
        # Experiment 2: Cross-stock prediction (train on first stock, test on others)
        train_stock = STOCKS[0]  # Use TLKM as training stock
        
        for iteration in range(n_iterations):
            print(f"Iteration {iteration + 1}/{n_iterations} (Training on {train_stock})")
            
            for test_stock in STOCKS:
                if train_stock == test_stock:
                    continue
                
                X_train, y_train, X_test, y_test, _ = prepare_cross_stock_data(
                    daily_data[train_stock], daily_data[test_stock],
                    train_ratio=TRAIN_RATIO, lookback=LOOKBACK
                )
                
                for model_type in MODEL_TYPES:
                    mape = train_single_model(X_train, y_train, X_test, y_test, model_type, iteration)
                    results.append({
                        'Iteration': iteration + 1,
                        'Train_Stock': train_stock,
                        'Test_Stock': test_stock,
                        'Stock': test_stock,
                        'Model': model_type,
                        'MAPE': mape
                    })
                    print(f"  {train_stock}→{test_stock} - {model_type}: MAPE = {mape:.4f}%" if not np.isnan(mape) else f"  {train_stock}→{test_stock} - {model_type}: FAILED")
            print()
    
    return pd.DataFrame(results)

## Run Experiment 1 (Same-Stock) with 5 Iterations

In [ ]:
exp1_results_5 = run_multiple_iterations('Exp1_80_20', ITERATIONS_5, is_same_stock=True)
print("\n✓ Experiment 1 - 5 iterations completed")
print(f"Results shape: {exp1_results_5.shape}")
print(exp1_results_5.head(10))

## Run Experiment 1 (Same-Stock) with 10 Iterations

In [ ]:
exp1_results_10 = run_multiple_iterations('Exp1_80_20', ITERATIONS_10, is_same_stock=True)
print("\n✓ Experiment 1 - 10 iterations completed")
print(f"Results shape: {exp1_results_10.shape}")
print(exp1_results_10.head(10))

## Run Experiment 2 (Cross-Stock) with 5 Iterations

In [ ]:
exp2_results_5 = run_multiple_iterations('Exp2_80_20', ITERATIONS_5, is_same_stock=False)
print("\n✓ Experiment 2 - 5 iterations completed")
print(f"Results shape: {exp2_results_5.shape}")
print(exp2_results_5.head(10))

In [ ]:
# Display Exp2 stock combinations and data information
print("\n" + "="*70)
print("EXPERIMENT 2 (5 iterations) - CROSS-STOCK PREDICTION")
print("="*70)
print(f"\nTraining Stock → Test Stocks:")
print(f"  TLKM → BBCA, ASII, UNVR")
print(f"\nStock Combinations Used:")
unique_pairs_5 = exp2_results_5[['Train_Stock', 'Test_Stock']].drop_duplicates()
for idx, row in unique_pairs_5.iterrows():
    train_size = len(daily_data[row['Train_Stock']]) * TRAIN_RATIO
    test_size = len(daily_data[row['Test_Stock']]) * (1 - TRAIN_RATIO)
    print(f"  {row['Train_Stock']} (train: {int(train_size)} samples) → {row['Test_Stock']} (test: {int(test_size)} samples)")
print(f"\nDetailed Results (first 20 rows):")
print(exp2_results_5.head(20))

## Run Experiment 2 (Cross-Stock) with 10 Iterations

In [ ]:
exp2_results_10 = run_multiple_iterations('Exp2_80_20', ITERATIONS_10, is_same_stock=False)
print("\n✓ Experiment 2 - 10 iterations completed")
print(f"Results shape: {exp2_results_10.shape}")
print(exp2_results_10.head(10))

In [ ]:
# Display Exp2 stock combinations and data information
print("\n" + "="*70)
print("EXPERIMENT 2 (10 iterations) - CROSS-STOCK PREDICTION")
print("="*70)
print(f"\nTraining Stock → Test Stocks:")
print(f"  TLKM → BBCA, ASII, UNVR")
print(f"\nStock Combinations Used:")
unique_pairs_10 = exp2_results_10[['Train_Stock', 'Test_Stock']].drop_duplicates()
for idx, row in unique_pairs_10.iterrows():
    train_size = len(daily_data[row['Train_Stock']]) * TRAIN_RATIO
    test_size = len(daily_data[row['Test_Stock']]) * (1 - TRAIN_RATIO)
    print(f"  {row['Train_Stock']} (train: {int(train_size)} samples) → {row['Test_Stock']} (test: {int(test_size)} samples)")
print(f"\nDetailed Results (first 20 rows):")
print(exp2_results_10.head(20))

## Compile and Prepare Results for Friedman Test

In [ ]:
# Add experiment labels
exp1_results_5['Experiment'] = 'Exp1_5it'
exp1_results_10['Experiment'] = 'Exp1_10it'
exp2_results_5['Experiment'] = 'Exp2_5it'
exp2_results_10['Experiment'] = 'Exp2_10it'

# Compile all results
all_results = pd.concat([
    exp1_results_5, exp1_results_10,
    exp2_results_5, exp2_results_10
], ignore_index=True)

print("Combined Results Summary:")
print(f"Total records: {len(all_results)}")
print(f"Experiments: {all_results['Experiment'].unique()}")
print(f"Models: {all_results['Model'].unique()}")
print(f"\nFirst 10 rows:")
print(all_results.head(10))
print(f"\nStatistics by Model:")
print(all_results.groupby('Model')['MAPE'].describe())

## Perform Friedman Test for Experiment 1

In [ ]:
# Separate 5 and 10 iterations for Experiment 1
exp1_5it_data = exp1_results_5.dropna()
exp1_10it_data = exp1_results_10.dropna()

print("="*70)
print("FRIEDMAN TEST - EXPERIMENT 1 (Same-Stock Prediction)")
print("="*70)

# Prepare data for Friedman test (5 iterations)
print(f"\n{'─'*70}")
print("Friedman Test - Experiment 1 with 5 Iterations")
print(f"{'─'*70}")

# Create pivot table: rows = samples (Stock×Iteration), columns = Models
exp1_5it_pivot = exp1_5it_data.pivot_table(
    index=['Stock', 'Iteration'],
    columns='Model',
    values='MAPE'
).reset_index(drop=True)

print(f"Sample size: {len(exp1_5it_pivot)}")
print(f"Models compared: {list(exp1_5it_pivot.columns)}")
print(f"\nPivot table (first 10 rows):")
print(exp1_5it_pivot.head(10))

# Perform Friedman test
groups_5 = [exp1_5it_pivot[col].values for col in exp1_5it_pivot.columns]
stat_5, p_value_5 = friedmanchisquare(*groups_5)

print(f"\n📊 Friedman Test Results (5 iterations):")
print(f"  Test Statistic: {stat_5:.4f}")
print(f"  P-value: {p_value_5:.6f}")
print(f"  Significance Level (α): 0.05")

if p_value_5 < 0.05:
    print(f"  ✓ SIGNIFICANT: Models show statistically significant differences (p < 0.05)")
else:
    print(f"  ✗ NOT SIGNIFICANT: No statistically significant differences (p >= 0.05)")

# Prepare data for Friedman test (10 iterations)
print(f"\n{'─'*70}")
print("Friedman Test - Experiment 1 with 10 Iterations")
print(f"{'─'*70}")

exp1_10it_pivot = exp1_10it_data.pivot_table(
    index=['Stock', 'Iteration'],
    columns='Model',
    values='MAPE'
).reset_index(drop=True)

print(f"Sample size: {len(exp1_10it_pivot)}")
print(f"Models compared: {list(exp1_10it_pivot.columns)}")
print(f"\nPivot table (first 10 rows):")
print(exp1_10it_pivot.head(10))

# Perform Friedman test
groups_10 = [exp1_10it_pivot[col].values for col in exp1_10it_pivot.columns]
stat_10, p_value_10 = friedmanchisquare(*groups_10)

print(f"\n📊 Friedman Test Results (10 iterations):")
print(f"  Test Statistic: {stat_10:.4f}")
print(f"  P-value: {p_value_10:.6f}")
print(f"  Significance Level (α): 0.05")

if p_value_10 < 0.05:
    print(f"  ✓ SIGNIFICANT: Models show statistically significant differences (p < 0.05)")
else:
    print(f"  ✗ NOT SIGNIFICANT: No statistically significant differences (p >= 0.05)")

## Perform Friedman Test for Experiment 2

In [ ]:
# Separate 5 and 10 iterations for Experiment 2
exp2_5it_data = exp2_results_5.dropna()
exp2_10it_data = exp2_results_10.dropna()

print("\n" + "="*70)
print("FRIEDMAN TEST - EXPERIMENT 2 (Cross-Stock Prediction)")
print("="*70)

# Prepare data for Friedman test (5 iterations)
print(f"\n{'─'*70}")
print("Friedman Test - Experiment 2 with 5 Iterations")
print(f"{'─'*70}")

exp2_5it_pivot = exp2_5it_data.pivot_table(
    index=['Stock', 'Iteration'],
    columns='Model',
    values='MAPE'
).reset_index(drop=True)

print(f"Sample size: {len(exp2_5it_pivot)}")
print(f"Models compared: {list(exp2_5it_pivot.columns)}")
print(f"\nPivot table (first 10 rows):")
print(exp2_5it_pivot.head(10))

# Perform Friedman test
groups_exp2_5 = [exp2_5it_pivot[col].values for col in exp2_5it_pivot.columns]
stat_exp2_5, p_value_exp2_5 = friedmanchisquare(*groups_exp2_5)

print(f"\n📊 Friedman Test Results (5 iterations):")
print(f"  Test Statistic: {stat_exp2_5:.4f}")
print(f"  P-value: {p_value_exp2_5:.6f}")
print(f"  Significance Level (α): 0.05")

if p_value_exp2_5 < 0.05:
    print(f"  ✓ SIGNIFICANT: Models show statistically significant differences (p < 0.05)")
else:
    print(f"  ✗ NOT SIGNIFICANT: No statistically significant differences (p >= 0.05)")

# Prepare data for Friedman test (10 iterations)
print(f"\n{'─'*70}")
print("Friedman Test - Experiment 2 with 10 Iterations")
print(f"{'─'*70}")

exp2_10it_pivot = exp2_10it_data.pivot_table(
    index=['Stock', 'Iteration'],
    columns='Model',
    values='MAPE'
).reset_index(drop=True)

print(f"Sample size: {len(exp2_10it_pivot)}")
print(f"Models compared: {list(exp2_10it_pivot.columns)}")
print(f"\nPivot table (first 10 rows):")
print(exp2_10it_pivot.head(10))

# Perform Friedman test
groups_exp2_10 = [exp2_10it_pivot[col].values for col in exp2_10it_pivot.columns]
stat_exp2_10, p_value_exp2_10 = friedmanchisquare(*groups_exp2_10)

print(f"\n📊 Friedman Test Results (10 iterations):")
print(f"  Test Statistic: {stat_exp2_10:.4f}")
print(f"  P-value: {p_value_exp2_10:.6f}")
print(f"  Significance Level (α): 0.05")

if p_value_exp2_10 < 0.05:
    print(f"  ✓ SIGNIFICANT: Models show statistically significant differences (p < 0.05)")
else:
    print(f"  ✗ NOT SIGNIFICANT: No statistically significant differences (p >= 0.05)")

## Summary of Friedman Test Results

In [ ]:
# Create summary DataFrame
friedman_summary = pd.DataFrame({
    'Experiment': ['Exp1 (Same-Stock)', 'Exp1 (Same-Stock)', 'Exp2 (Cross-Stock)', 'Exp2 (Cross-Stock)'],
    'Iterations': [5, 10, 5, 10],
    'Test Statistic': [stat_5, stat_10, stat_exp2_5, stat_exp2_10],
    'P-value': [p_value_5, p_value_10, p_value_exp2_5, p_value_exp2_10],
    'Significant (α=0.05)': [
        'YES' if p_value_5 < 0.05 else 'NO',
        'YES' if p_value_10 < 0.05 else 'NO',
        'YES' if p_value_exp2_5 < 0.05 else 'NO',
        'YES' if p_value_exp2_10 < 0.05 else 'NO'
    ]
})

print("\n" + "="*70)
print("FRIEDMAN TEST SUMMARY")
print("="*70)
print(friedman_summary.to_string(index=False))

# Save summary
friedman_summary.to_csv('results/friedman_test_summary.csv', index=False)
print("\n✓ Summary saved to: results/friedman_test_summary.csv")

## Visualization: MAPE Distribution by Model (Experiment 1)

In [ ]:
# Create box plot for Experiment 1 (5 iterations)
fig_exp1_5 = px.box(
    exp1_results_5,
    x='Model',
    y='MAPE',
    color='Model',
    title='Experiment 1 (Same-Stock) - MAPE Distribution (5 iterations)',
    color_discrete_map=MODEL_COLORS,
    labels={'MAPE': 'MAPE (%)', 'Model': 'Model'},
    points='all'
)
fig_exp1_5.update_layout(height=600, width=1000)
fig_exp1_5.write_html('figures/friedman_test/Exp1_5it_box_plot.html', config=PLOTLY_HTML_CONFIG)
fig_exp1_5.show()
print("✓ Saved: figures/friedman_test/Exp1_5it_box_plot.html")

# Create box plot for Experiment 1 (10 iterations)
fig_exp1_10 = px.box(
    exp1_results_10,
    x='Model',
    y='MAPE',
    color='Model',
    title='Experiment 1 (Same-Stock) - MAPE Distribution (10 iterations)',
    color_discrete_map=MODEL_COLORS,
    labels={'MAPE': 'MAPE (%)', 'Model': 'Model'},
    points='all'
)
fig_exp1_10.update_layout(height=600, width=1000)
fig_exp1_10.write_html('figures/friedman_test/Exp1_10it_box_plot.html', config=PLOTLY_HTML_CONFIG)
fig_exp1_10.show()
print("✓ Saved: figures/friedman_test/Exp1_10it_box_plot.html")

## Visualization: MAPE Distribution by Model (Experiment 2)

In [ ]:
# Create box plot for Experiment 2 (5 iterations)
fig_exp2_5 = px.box(
    exp2_results_5,
    x='Model',
    y='MAPE',
    color='Model',
    title='Experiment 2 (Cross-Stock) - MAPE Distribution (5 iterations)',
    color_discrete_map=MODEL_COLORS,
    labels={'MAPE': 'MAPE (%)', 'Model': 'Model'},
    points='all'
)
fig_exp2_5.update_layout(height=600, width=1000)
fig_exp2_5.write_html('figures/friedman_test/Exp2_5it_box_plot.html', config=PLOTLY_HTML_CONFIG)
fig_exp2_5.show()
print("✓ Saved: figures/friedman_test/Exp2_5it_box_plot.html")

# Create box plot for Experiment 2 (10 iterations)
fig_exp2_10 = px.box(
    exp2_results_10,
    x='Model',
    y='MAPE',
    color='Model',
    title='Experiment 2 (Cross-Stock) - MAPE Distribution (10 iterations)',
    color_discrete_map=MODEL_COLORS,
    labels={'MAPE': 'MAPE (%)', 'Model': 'Model'},
    points='all'
)
fig_exp2_10.update_layout(height=600, width=1000)
fig_exp2_10.write_html('figures/friedman_test/Exp2_10it_box_plot.html', config=PLOTLY_HTML_CONFIG)
fig_exp2_10.show()
print("✓ Saved: figures/friedman_test/Exp2_10it_box_plot.html")

## Visualization: MAPE Trends Across Iterations

In [ ]:
# Average MAPE by model and iteration (Exp1)
exp1_agg_5 = exp1_results_5.groupby(['Iteration', 'Model'])['MAPE'].mean().reset_index()
fig_trend_exp1_5 = px.line(
    exp1_agg_5,
    x='Iteration',
    y='MAPE',
    color='Model',
    color_discrete_map=MODEL_COLORS,
    title='Experiment 1 (5 iterations) - Average MAPE per Iteration',
    markers=True,
    labels={'MAPE': 'Average MAPE (%)', 'Iteration': 'Iteration'}
)
fig_trend_exp1_5.update_layout(height=600, width=1000)
fig_trend_exp1_5.write_html('figures/friedman_test/Exp1_5it_trends.html', config=PLOTLY_HTML_CONFIG)
fig_trend_exp1_5.show()
print("✓ Saved: figures/friedman_test/Exp1_5it_trends.html")

# Average MAPE by model and iteration (Exp1 - 10 iterations)
exp1_agg_10 = exp1_results_10.groupby(['Iteration', 'Model'])['MAPE'].mean().reset_index()
fig_trend_exp1_10 = px.line(
    exp1_agg_10,
    x='Iteration',
    y='MAPE',
    color='Model',
    color_discrete_map=MODEL_COLORS,
    title='Experiment 1 (10 iterations) - Average MAPE per Iteration',
    markers=True,
    labels={'MAPE': 'Average MAPE (%)', 'Iteration': 'Iteration'}
)
fig_trend_exp1_10.update_layout(height=600, width=1000)
fig_trend_exp1_10.write_html('figures/friedman_test/Exp1_10it_trends.html', config=PLOTLY_HTML_CONFIG)
fig_trend_exp1_10.show()
print("✓ Saved: figures/friedman_test/Exp1_10it_trends.html")

# Average MAPE by model and iteration (Exp2)
exp2_agg_5 = exp2_results_5.groupby(['Iteration', 'Model'])['MAPE'].mean().reset_index()
fig_trend_exp2_5 = px.line(
    exp2_agg_5,
    x='Iteration',
    y='MAPE',
    color='Model',
    color_discrete_map=MODEL_COLORS,
    title='Experiment 2 (5 iterations) - Average MAPE per Iteration',
    markers=True,
    labels={'MAPE': 'Average MAPE (%)', 'Iteration': 'Iteration'}
)
fig_trend_exp2_5.update_layout(height=600, width=1000)
fig_trend_exp2_5.write_html('figures/friedman_test/Exp2_5it_trends.html', config=PLOTLY_HTML_CONFIG)
fig_trend_exp2_5.show()
print("✓ Saved: figures/friedman_test/Exp2_5it_trends.html")

# Average MAPE by model and iteration (Exp2 - 10 iterations)
exp2_agg_10 = exp2_results_10.groupby(['Iteration', 'Model'])['MAPE'].mean().reset_index()
fig_trend_exp2_10 = px.line(
    exp2_agg_10,
    x='Iteration',
    y='MAPE',
    color='Model',
    color_discrete_map=MODEL_COLORS,
    title='Experiment 2 (10 iterations) - Average MAPE per Iteration',
    markers=True,
    labels={'MAPE': 'Average MAPE (%)', 'Iteration': 'Iteration'}
)
fig_trend_exp2_10.update_layout(height=600, width=1000)
fig_trend_exp2_10.write_html('figures/friedman_test/Exp2_10it_trends.html', config=PLOTLY_HTML_CONFIG)
fig_trend_exp2_10.show()
print("✓ Saved: figures/friedman_test/Exp2_10it_trends.html")

## Visualization: Statistical Comparison Dashboard

In [ ]:
# Create statistical comparison dashboard
fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Exp1 (5it) - P-value',
        'Exp1 (10it) - P-value',
        'Exp2 (5it) - P-value',
        'Exp2 (10it) - P-value'
    ),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# Threshold line at 0.05
threshold = 0.05

# Add bars with colors indicating significance
colors_sig = ['green' if p < threshold else 'red' for p in 
              [p_value_5, p_value_10, p_value_exp2_5, p_value_exp2_10]]

fig_dashboard.add_trace(
    go.Bar(x=['Exp1 (5it)'], y=[p_value_5], marker_color=colors_sig[0], name='P-value'),
    row=1, col=1
)
fig_dashboard.add_hline(y=0.05, line_dash="dash", line_color="gray", row=1, col=1)
fig_dashboard.add_annotation(text="α=0.05", x=0.5, y=0.06, xref="x", yref="y", 
                             showarrow=False, row=1, col=1)

fig_dashboard.add_trace(
    go.Bar(x=['Exp1 (10it)'], y=[p_value_10], marker_color=colors_sig[1], name='P-value'),
    row=1, col=2
)
fig_dashboard.add_hline(y=0.05, line_dash="dash", line_color="gray", row=1, col=2)
fig_dashboard.add_annotation(text="α=0.05", x=0.5, y=0.06, xref="x2", yref="y2", 
                             showarrow=False, row=1, col=2)

fig_dashboard.add_trace(
    go.Bar(x=['Exp2 (5it)'], y=[p_value_exp2_5], marker_color=colors_sig[2], name='P-value'),
    row=2, col=1
)
fig_dashboard.add_hline(y=0.05, line_dash="dash", line_color="gray", row=2, col=1)
fig_dashboard.add_annotation(text="α=0.05", x=0.5, y=0.06, xref="x3", yref="y3", 
                             showarrow=False, row=2, col=1)

fig_dashboard.add_trace(
    go.Bar(x=['Exp2 (10it)'], y=[p_value_exp2_10], marker_color=colors_sig[3], name='P-value'),
    row=2, col=2
)
fig_dashboard.add_hline(y=0.05, line_dash="dash", line_color="gray", row=2, col=2)
fig_dashboard.add_annotation(text="α=0.05", x=0.5, y=0.06, xref="x4", yref="y4", 
                             showarrow=False, row=2, col=2)

fig_dashboard.update_yaxes(title_text="P-value", row=1, col=1)
fig_dashboard.update_yaxes(title_text="P-value", row=1, col=2)
fig_dashboard.update_yaxes(title_text="P-value", row=2, col=1)
fig_dashboard.update_yaxes(title_text="P-value", row=2, col=2)

fig_dashboard.update_layout(height=800, width=1200, showlegend=False, 
                           title_text="Friedman Test Results - Statistical Significance (Green=Significant, Red=Not Significant)")
fig_dashboard.write_html('figures/friedman_test/friedman_dashboard.html', config=PLOTLY_HTML_CONFIG)
fig_dashboard.show()
print("✓ Saved: figures/friedman_test/friedman_dashboard.html")

## Detailed Statistical Summary by Model

In [ ]:
print("\n" + "="*70)
print("DETAILED MAPE STATISTICS BY MODEL")
print("="*70)

# Experiment 1
print("\n" + "─"*70)
print("EXPERIMENT 1 - SAME-STOCK PREDICTION")
print("─"*70)

print("\n5 Iterations:")
stats_exp1_5 = exp1_results_5.groupby('Model')['MAPE'].agg(['mean', 'std', 'min', 'max', 'median']).round(4)
print(stats_exp1_5)

print("\n10 Iterations:")
stats_exp1_10 = exp1_results_10.groupby('Model')['MAPE'].agg(['mean', 'std', 'min', 'max', 'median']).round(4)
print(stats_exp1_10)

# Experiment 2
print("\n" + "─"*70)
print("EXPERIMENT 2 - CROSS-STOCK PREDICTION")
print("─"*70)

print("\n5 Iterations:")
stats_exp2_5 = exp2_results_5.groupby('Model')['MAPE'].agg(['mean', 'std', 'min', 'max', 'median']).round(4)
print(stats_exp2_5)

print("\n10 Iterations:")
stats_exp2_10 = exp2_results_10.groupby('Model')['MAPE'].agg(['mean', 'std', 'min', 'max', 'median']).round(4)
print(stats_exp2_10)

# Save detailed statistics
stats_exp1_5.to_csv('results/exp1_5it_stats.csv')
stats_exp1_10.to_csv('results/exp1_10it_stats.csv')
stats_exp2_5.to_csv('results/exp2_5it_stats.csv')
stats_exp2_10.to_csv('results/exp2_10it_stats.csv')

print("\n✓ Statistical summaries saved to results/")

## Interpretation and Conclusions

### What is the Friedman Test?

The **Friedman Test** is a non-parametric statistical test used to:
- Compare multiple related samples (matched/repeated measurements)
- Test if there are significant differences between treatments
- **Advantage:** Does not assume normal distribution of data
- **Ideal for:** Multiple models tested on same data with multiple iterations

**Null Hypothesis (H₀):** All models have similar performance (no significant differences)  
**Alternative Hypothesis (H₁):** At least one model has significantly different performance

**Decision Rule:**
- If **p-value < 0.05** → REJECT H₀ → Significant differences exist
- If **p-value ≥ 0.05** → FAIL TO REJECT H₀ → No significant differences

### Results Interpretation

#### **Experiment 1: Same-Stock Prediction (80/20)**
The model trains on each stock's historical data and predicts that same stock's future prices.

- **5 Iterations:** p-value = {:.6f} {}
- **10 Iterations:** p-value = {:.6f} {}

#### **Experiment 2: Cross-Stock Prediction (80/20)**
The model trains on one stock (TLKM) and predicts different stocks' future prices (zero-shot transfer).

- **5 Iterations:** p-value = {:.6f} {}
- **10 Iterations:** p-value = {:.6f} {}

### Key Findings

1. **Model Consistency:** Results from 5 and 10 iterations are compared to assess consistency.
2. **Statistical Significance:** Green boxes indicate significant differences; Red indicates no significant differences.
3. **MAPE Values:** Lower MAPE values indicate better prediction accuracy (error-based metric).
4. **Model Rankings:** Check the statistics table to see which model performs best on average.

### Recommendations

- If differences ARE significant: Use the best-performing model with lowest mean MAPE
- If differences are NOT significant: Choose based on training time, complexity, or other criteria
- Consider the trade-off between accuracy and computational efficiency
- Bidirectional models (BiLSTM, BiGRU) typically provide better predictions but require more computation

""".format(p_value_5, "✓ SIGNIFICANT" if p_value_5 < 0.05 else "✗ NOT SIGNIFICANT",
           p_value_10, "✓ SIGNIFICANT" if p_value_10 < 0.05 else "✗ NOT SIGNIFICANT",
           p_value_exp2_5, "✓ SIGNIFICANT" if p_value_exp2_5 < 0.05 else "✗ NOT SIGNIFICANT",
           p_value_exp2_10, "✓ SIGNIFICANT" if p_value_exp2_10 < 0.05 else "✗ NOT SIGNIFICANT")
)

### Summary Table

In [ ]:
# Create final summary table
final_summary = pd.DataFrame({
    'Experiment': ['Exp1 - Same-Stock', 'Exp1 - Same-Stock', 'Exp2 - Cross-Stock', 'Exp2 - Cross-Stock'],
    'Iterations': ['5', '10', '5', '10'],
    'P-Value': [f'{p_value_5:.6f}', f'{p_value_10:.6f}', f'{p_value_exp2_5:.6f}', f'{p_value_exp2_10:.6f}'],
    'Test Statistic': [f'{stat_5:.4f}', f'{stat_10:.4f}', f'{stat_exp2_5:.4f}', f'{stat_exp2_10:.4f}'],
    'Result': [
        '✓ SIGNIFICANT' if p_value_5 < 0.05 else '✗ NOT SIGNIFICANT',
        '✓ SIGNIFICANT' if p_value_10 < 0.05 else '✗ NOT SIGNIFICANT',
        '✓ SIGNIFICANT' if p_value_exp2_5 < 0.05 else '✗ NOT SIGNIFICANT',
        '✓ SIGNIFICANT' if p_value_exp2_10 < 0.05 else '✗ NOT SIGNIFICANT'
    ]
})

print("\n" + "="*70)
print("FRIEDMAN TEST - FINAL SUMMARY")
print("="*70)
print(final_summary.to_string(index=False))

print("\n\n" + "="*70)
print("OVERALL CONCLUSIONS")
print("="*70)
print("""
1. EXPERIMENT 1 (Same-Stock Prediction):
   - Training on a stock's own data to predict its future prices
   - 5 iterations shows p-value = {:.6f} ({})
   - 10 iterations shows p-value = {:.6f} ({})
   
2. EXPERIMENT 2 (Cross-Stock Prediction):
   - Zero-shot transfer: train on one stock, test on another
   - 5 iterations shows p-value = {:.6f} ({})
   - 10 iterations shows p-value = {:.6f} ({})

3. BEST PERFORMING MODELS (by average MAPE):
   Exp1-5it:  {} (μ={:.4f}%)
   Exp1-10it: {} (μ={:.4f}%)
   Exp2-5it:  {} (μ={:.4f}%)
   Exp2-10it: {} (μ={:.4f}%)
""".format(
    p_value_5, "SIGNIFICANT" if p_value_5 < 0.05 else "NOT SIGNIFICANT",
    p_value_10, "SIGNIFICANT" if p_value_10 < 0.05 else "NOT SIGNIFICANT",
    p_value_exp2_5, "SIGNIFICANT" if p_value_exp2_5 < 0.05 else "NOT SIGNIFICANT",
    p_value_exp2_10, "SIGNIFICANT" if p_value_exp2_10 < 0.05 else "NOT SIGNIFICANT",
    stats_exp1_5['mean'].idxmin(), stats_exp1_5['mean'].min(),
    stats_exp1_10['mean'].idxmin(), stats_exp1_10['mean'].min(),
    stats_exp2_5['mean'].idxmin(), stats_exp2_5['mean'].min(),
    stats_exp2_10['mean'].idxmin(), stats_exp2_10['mean'].min()
))

# Save final summary
final_summary.to_csv('results/friedman_test_final_summary.csv', index=False)
print("✓ Final summary saved to: results/friedman_test_final_summary.csv\n")

## Export Complete Results to CSV

In [ ]:
# Export all results
exp1_results_5.to_csv('results/friedman_exp1_5it_results.csv', index=False)
exp1_results_10.to_csv('results/friedman_exp1_10it_results.csv', index=False)
exp2_results_5.to_csv('results/friedman_exp2_5it_results.csv', index=False)
exp2_results_10.to_csv('results/friedman_exp2_10it_results.csv', index=False)
all_results.to_csv('results/friedman_all_results_combined.csv', index=False)

print("\n" + "="*70)
print("FILES GENERATED")
print("="*70)
print("""
CSV Results Files:
  ✓ results/friedman_exp1_5it_results.csv - Exp1 with 5 iterations
  ✓ results/friedman_exp1_10it_results.csv - Exp1 with 10 iterations
  ✓ results/friedman_exp2_5it_results.csv - Exp2 with 5 iterations
  ✓ results/friedman_exp2_10it_results.csv - Exp2 with 10 iterations
  ✓ results/friedman_all_results_combined.csv - All combined results
  ✓ results/friedman_test_summary.csv - Friedman test summary
  ✓ results/friedman_test_final_summary.csv - Final conclusions

Statistics Files:
  ✓ results/exp1_5it_stats.csv - Exp1 5it MAPE statistics
  ✓ results/exp1_10it_stats.csv - Exp1 10it MAPE statistics
  ✓ results/exp2_5it_stats.csv - Exp2 5it MAPE statistics
  ✓ results/exp2_10it_stats.csv - Exp2 10it MAPE statistics

Interactive Visualizations (HTML):
  ✓ figures/friedman_test/Exp1_5it_box_plot.html
  ✓ figures/friedman_test/Exp1_10it_box_plot.html
  ✓ figures/friedman_test/Exp2_5it_box_plot.html
  ✓ figures/friedman_test/Exp2_10it_box_plot.html
  ✓ figures/friedman_test/Exp1_5it_trends.html
  ✓ figures/friedman_test/Exp1_10it_trends.html
  ✓ figures/friedman_test/Exp2_5it_trends.html
  ✓ figures/friedman_test/Exp2_10it_trends.html
  ✓ figures/friedman_test/friedman_dashboard.html
""")

print("="*70)
print("✓ FRIEDMAN TEST ANALYSIS COMPLETE!")
print("="*70)